In [2]:
%pip install -U google-generativeai
%pip install google-ai-generativelanguage==0.6.15
%pip install -U langchain-google-genai
%pip install -U langchain-community
%pip install -U langgraph
%pip install -U langgraph langchain-community
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [13]:
import os 
from dotenv import load_dotenv
import re 
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, List
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from tavily import TavilyClient


In [14]:
load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')

In [15]:
model = ChatOpenAI(
    model='gpt-5-nano',
    temperature=0.2,
    api_key=os.getenv('OPENAI_API_KEY')
)

In [16]:
client = TavilyClient(api_key=os.environ.get('TAVILY_API_KEY'))

result = client.search("O que sao multiAgentes de Inteligencia Artificial?", include_answer=True)

result["answer"]

'Sistemas multiagentes de IA são conjuntos de agentes de IA que colaboram para resolver problemas complexos. Cada agente tem funções específicas e trabalha em conjunto. Eles são usados para tarefas que exigem interação e compartilhamento de informações entre múltiplas inteligências artificiais.'

In [19]:
cidade = "Belém"

query = f"""
Liste os 5 principais restaurantes em {cidade}, segundo avaliações recentes no TripAdvisor ou sites similares de turismo.
Para cada restaurante, informe:
- Tipo de culinária (ex: regional, italiana, japonesa)
- Uma breve descrição (máx. 2 linhas)
- Avaliação média (se disponível)
- Faixa de preço

Responda apenas com dados atualizados e relevantes para turistas.
"""

In [17]:
%pip install -u duckduckgo_search


[optparse.groups]Usage:[/]   
  /Users/lucasarruda/Study/Alura/LangGraph_Orquestrando_agentes/.venv/bin/python -m pip install \[options] <requirement specifier> \[package-index-options] ...
  /Users/lucasarruda/Study/Alura/LangGraph_Orquestrando_agentes/.venv/bin/python -m pip install \[options] -r <requirements file> \[package-index-options] ...
  /Users/lucasarruda/Study/Alura/LangGraph_Orquestrando_agentes/.venv/bin/python -m pip install \[options] [-e] <vcs project url> ...
  /Users/lucasarruda/Study/Alura/LangGraph_Orquestrando_agentes/.venv/bin/python -m pip install \[options] [-e] <local project path> ...
  /Users/lucasarruda/Study/Alura/LangGraph_Orquestrando_agentes/.venv/bin/python -m pip install \[options] <archive url/path> ...

no such option: -u
Note: you may need to restart the kernel to use updated packages.


In [21]:
import requests
from bs4 import BeautifulSoup
from duckduckgo_search import DDGS
import re


ddg = DDGS()

def search(query, max_results=6):
    try:
        results = ddg.text(query, max_results=max_results)
        return [i["href"] for i in results]
    except Exception as e:
        raise e
    
for link in search(query):
        print(link)

/var/folders/gx/9vrjjlfd295c01q_1m9vd9kc0000gn/T/ipykernel_10679/2821713512.py:7: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  ddg = DDGS()


https://www.tripadvisor.com.br/Restaurants-g303404-Belem_State_of_Para.html
https://www.tripadvisor.com.br/Restaurants-g303404-zfg16548-Belem_State_of_Para.html
https://www.tripadvisor.pt/Restaurants-g303404-Belem_State_of_Para.html
https://veja.abril.com.br/comer-e-beber/belem/comer-e-beber-belem-2025-restaurantes-melhores/
https://www.tripadvisor.pt/Restaurants-g303404-zfp10954-Belem_State_of_Para.html
https://exame.com/casual/os-melhores-restaurantes-de-belem-segundo-ranking-exame-casual-2025/


In [24]:
import re 
from tavily import TavilyClient

api_key = os.environ.get("TAVILY_API_KEY")

cliente_tavily = TavilyClient(api_key=api_key)

cidade = "Belem do Para"
tavily_query = f"Restaurantes em {cidade} tripadvisor com maior quantidade de reviews e faixa de preco"


tripadvisor_url = None

try:
    tavily_results = cliente_tavily.search(query=tavily_query, max_results=5)

    if tavily_results and tavily_results["results"]:
        print(f"tavily encontrou {len(tavily_results['results'])} resultados. Analisando....")

        for result in tavily_results["results"]:
            url = result["url"]

            if "tripadvisor.com" in url or "tripadvisor.com.br" in url:
                tripadvisor_url = url
                break

        if not tripadvisor_url:
            print("Nenhum URL relevante do Tripadivsor foi encontrado nos primeiros resultados")
    else:
        print("Tavily nao encontrou resultados para a busca agentica")


except Exception as e: 
    raise e

if tripadvisor_url:
    clean_url = re.sub(r'-oa\d+-', '-', tripadvisor_url)
    tripadvisor_url = clean_url
    print(f"URL Encontrada limpa de paginacao")

print('-' * 50)
print(f"URL Final do tripadvisor para raspagem: {tripadvisor_url if tripadvisor_url else 'NAO ENCONTRADO'}")
print('-' * 50)


tavily encontrou 5 resultados. Analisando....
URL Encontrada limpa de paginacao
--------------------------------------------------
URL Final do tripadvisor para raspagem: https://www.tripadvisor.com/Restaurants-g303404-zfg16556-Belem_State_of_Para.html
--------------------------------------------------


In [25]:
%pip install selenium
%pip install webdriver-manager

/opt/homebrew/Cellar/python@3.12/3.12.13_2/Frameworks/Python.framework/Versions/3.12/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=10679) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 11.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [selenium]6/7 [selenium]
Note: you may need to restart the kernel to use updated packages.


/opt/homebrew/Cellar/python@3.12/3.12.13_2/Frameworks/Python.framework/Versions/3.12/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=10679) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


Note: you may need to restart the kernel to use updated packages.


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
from selenium.common.exceptions import WebDriverException, TimeoutException

def scrape_restaurantes_info(url):
    if not url:
        print("Erro: URL vazia ou não localizada para raspagem.")
        return None

    try:
        service = Service(ChromeDriverManager().install())
        options = webdriver.ChromeOptions()
        options.add_argument("--headless")
        options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36")
        
        driver = webdriver.Chrome(service=service, options=options)
        driver.set_page_load_timeout(30)
        
    except Exception as e:
        print(f"Erro ao inicializar o driver do Selenium: {e}")
        return None

    try:
        print(f"Tentando carregar a página com Selenium: {url}")
        driver.get(url)
        
        driver.implicitly_wait(10)

        response_text = driver.page_source

    except TimeoutException:
        print(f"Erro de tempo limite ao carregar a página {url}.")
        return None
    except WebDriverException as e:
        print(f"Erro ao carregar a página {url} com Selenium: {e}. Pode ser um bloqueio ou problema de conexão.")
        return None
    finally:
        driver.quit()

    soup = BeautifulSoup(response_text, 'html.parser')
    return soup

soup_tripadvisor = None

if 'tripadvisor_url' in locals() and tripadvisor_url:
    print(f"\nTentando raspar a página identificada: {tripadvisor_url}")
    soup_tripadvisor = scrape_restaurantes_info(tripadvisor_url)

    if soup_tripadvisor:
        print("HTML da página do Tripadvisor obtido com sucesso!")
        page_title_tag = soup_tripadvisor.find('title')
        if page_title_tag:
            print(f"Título da página: {page_title_tag.get_text(strip=True)}")
        else:
            print("Não foi possível encontrar o título da página.")
    else:
        print("Falha ao raspar a página do Tripadvisor. Verifique o URL ou se o site está bloqueando.")
else:
    print("Não há URL do Tripadvisor válido para raspar (obtido no Bloco 1).")

print("-" * 50)